In [2]:
import numpy as np
import pandas as pd

In [8]:
df = pd.read_csv('dataset.csv')

In [9]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  str    
 1   availability  13320 non-null  str    
 2   location      13319 non-null  str    
 3   size          13304 non-null  str    
 4   society       7818 non-null   str    
 5   total_sqft    13320 non-null  str    
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), str(6)
memory usage: 936.7 KB


In [11]:
df['area_type'].value_counts()

area_type
Super built-up  Area    8790
Built-up  Area          2418
Plot  Area              2025
Carpet  Area              87
Name: count, dtype: int64

In [12]:
df['size'].value_counts()

size
2 BHK         5199
3 BHK         4310
4 Bedroom      826
4 BHK          591
3 Bedroom      547
1 BHK          538
2 Bedroom      329
5 Bedroom      297
6 Bedroom      191
1 Bedroom      105
8 Bedroom       84
7 Bedroom       83
5 BHK           59
9 Bedroom       46
6 BHK           30
7 BHK           17
1 RK            13
10 Bedroom      12
9 BHK            8
8 BHK            5
11 BHK           2
11 Bedroom       2
10 BHK           2
27 BHK           1
19 BHK           1
16 BHK           1
43 Bedroom       1
14 BHK           1
12 Bedroom       1
13 BHK           1
18 Bedroom       1
Name: count, dtype: int64

In [13]:
df['availability'].value_counts()

availability
Ready To Move    10581
18-Dec             307
18-May             295
18-Apr             271
18-Aug             200
                 ...  
15-Aug               1
17-Jan               1
16-Nov               1
16-Jan               1
14-Jul               1
Name: count, Length: 81, dtype: int64

In [15]:
df['bath'].value_counts()

bath
2.0     6908
3.0     3286
4.0     1226
1.0      788
5.0      524
6.0      273
7.0      102
8.0       64
9.0       43
10.0      13
12.0       7
11.0       3
13.0       3
16.0       2
14.0       1
27.0       1
40.0       1
15.0       1
18.0       1
Name: count, dtype: int64

In [14]:
# EDA finindings:
#0. Area_type is having just 4 categories and that would affect a lot along with the total_sqft as plot_area category with total_sqft would have higher space and super builtup area with sq_feet
#1. availabilty has ready to move and specific dates. The specific dates can be simply converted to not ready to move. So we will have just 2 categories in it
#2. location has lots of categories... how to handle those many categories => 
#3. size also has bhk as well as bedroom setup like 2bhk, 2 bedrooms both are same thing and there are multiple such instances for 3bhk, 4, 5 and so on => maybe let's convert first all the 'bedroom' columns to bhk. Then there would still be some duplicate columns so those duplicate columns can be merged.
#4. society column has 5502 empty rows, how to fix that? => I think it won't be having a major correlation with the price. though it maybe possible that if a society belongs to a locality in that city and that locality may demand a premium, so there is still a chance but having so many null values meaning the feature maynot be explored by model?
#5. Total_sqft, I feel this will be the highly correlative feature to the target column which is price
#6. bath should be the number of bathroom the house has and 73 rows have null values => more number of bathrooms could mean more the price, so better to keep the null values as is
#7. balcony has 609 rows empty => balcony may have good correlation with price column so it's better to keep emtpy in the data since imputing any value can introduce bias?

In [16]:
df['location'].value_counts()

location
Whitefield                                         540
Sarjapur  Road                                     399
Electronic City                                    302
Kanakpura Road                                     273
Thanisandra                                        234
                                                  ... 
Pattegarhpalya                                       1
Tilak Nagar                                          1
12th cross srinivas nagar banshankari 3rd stage      1
Havanur extension                                    1
Abshot Layout                                        1
Name: count, Length: 1305, dtype: int64

In [17]:
df['location'].isnull().sum()

np.int64(1)

In [21]:
counts = df['location'].value_counts()

total_single = (counts == 1).sum()
print("how many are appeared just once", total_single)

above_ten = (counts > 10).sum()
print("how many are above 10 have appeared", above_ten)

how many are appeared just once 480
how many are above 10 have appeared 241


# 3 ways to handle the location issue ->
#since location has 1308 values we cannot use normal one-hot encoding since it would increase the columns to huge numbers
#1. frequency encoding replaces the frequency of that category with that actual categorical value. 
#2. in case there are so many categories but appearing just once then those can be categorised in others.
#3. Target encoding => This replaces each category with the average value for that specific category. And this actually makes sense since the houses specific to that location can be seen as an average of the target price.

#analysis :
#out of 1308 locations there are 480 locations which appears just once
#but there are still 241 categories which appeared more than 10 times. And that too is a huge number to handle with OHE, so can't use the 2nd way.


So, I have **1,308 localities**, where the most common locality appears **540 times**, the second most common appears **399 times**, and so on. Also, around **480 localities appear only once**.

with K-Fold Target Encoding, am I going to ignore those 540 rows of the most common locality when calculating its encoding?

Or, that I have to divide the entire dataset into **5 folds**? For example, if I have around **13,000 rows**, each fold will have around **2,600–2,700 rows**.

Then:

* **Fold 1** is encoded using the target values from Folds 2, 3, 4, and 5.
* **Fold 2** is encoded using Folds 1, 3, 4, and 5.
* **Fold 3** is encoded using Folds 1, 2, 4, and 5.
* And so on.

This way, **every row gets its target encoding calculated using other rows, not using its own fold**.

So, even for the locality that appears 540 times, I don't simply ignore those rows. I use the other folds to calculate its target encoding.

The main benefit is that the target value from a particular fold is **not used to encode that same fold**, which helps prevent **target leakage**.

Even the localities that appear only once are handled. If a locality appears only once in Fold 1, its encoding can be calculated from the other four folds. However, because it has very little data, its encoding may be less reliable.

The rare 480 single-occurrence locations are the interesting edge case — with only 1 row, 4 of the 5 folds will have zero rows for that location to calculate a mean from. The standard solution is a smoothing parameter:
Encoded value = (count × location_mean + global_mean × smoothing) / (count + smoothing)

In [22]:
df['bath'].isnull().sum()

np.int64(73)

In [25]:
df.loc[df['bath'].isna(), 'size']

56       4 Bedroom
81       4 Bedroom
224          3 BHK
344          1 BHK
579            NaN
           ...    
11496        1 BHK
11569          NaN
12768    5 Bedroom
12861        4 BHK
13240        1 BHK
Name: size, Length: 73, dtype: str

For handling the bathroom and balcony column I can take median of those associated sizes. for e.g the 2bhk median for bath across data
maybe 2 and so that would be applied to the null values. And this technique will be applied to both bath and balcony

In [28]:
#fixing the size column..
df['size_merged'] = df['size'].str.replace(r'(\d+)\s*bedroom', r'\1 BHK', case=False, regex=True)

In [30]:
df['size_merged'].isnull().sum()

np.int64(16)

In [31]:
df['size'].isnull().sum()

np.int64(16)

In [32]:
df[df['size'].isna()]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,size_merged
579,Plot Area,Immediate Possession,Sarjapur Road,NaN,Asiss B,1200 - 2400,NaN,NaN,34.185,NaN
1775,Plot Area,Immediate Possession,IVC Road,NaN,Orana N,2000 - 5634,NaN,NaN,124.000,NaN
2264,Plot Area,Immediate Possession,Banashankari,NaN,NaN,2400,NaN,NaN,460.000,NaN
2809,Plot Area,Immediate Possession,Sarjapur Road,NaN,AsdiaAr,1200 - 2400,NaN,NaN,28.785,NaN
2862,Plot Area,Immediate Possession,Devanahalli,NaN,Ajleyor,1500 - 2400,NaN,NaN,46.800,NaN
5333,Plot Area,Immediate Possession,Devanahalli,NaN,Emngs S,2100 - 5405,NaN,NaN,177.115,NaN
6423,Plot Area,Immediate Possession,Whitefield,NaN,SRniaGa,2324,NaN,NaN,26.730,NaN
6636,Plot Area,Immediate Possession,Jigani,NaN,S2enste,1500,NaN,NaN,25.490,NaN
6719,Plot Area,Immediate Possession,Hoskote,NaN,SJowsn,800 - 2660,NaN,NaN,28.545,NaN
7680,Plot Area,Immediate Possession,Kasavanhalli,NaN,NaN,5000,NaN,NaN,400.000,NaN


In [33]:
#important observations..
#1. size is null for 16 rows.
#2. the sqfeet is also is in the range